# BMC2025 - Notebook operativo

Notebook allineato a `pipeline.R` per pre-analisi e metriche iniziali

In [ ]:
options(stringsAsFactors = FALSE)\n
\n
needed <- c("haven", "dplyr", "tibble", "tidyr", "ggplot2", "caret", "rpart", "randomForest", "e1071")\n
missing <- needed[!vapply(needed, requireNamespace, logical(1), quietly = TRUE)]\n
if (length(missing) > 0) install.packages(missing)\n
\n
invisible(lapply(needed, library, character.only = TRUE))\n
set.seed(123)\n

In [ ]:
find_project_root <- function(start = getwd()) {\n
  current <- normalizePath(start, winslash = "/", mustWork = TRUE)\n
  repeat {\n
    has_pipeline <- file.exists(file.path(current, "pipeline.R"))\n
    has_data <- dir.exists(file.path(current, "data"))\n
    if (has_pipeline && has_data) return(current)\n
\n
    parent <- dirname(current)\n
    if (identical(parent, current)) stop("Project root non trovato (atteso pipeline.R + data/).")\n
    current <- parent\n
  }\n
}\n
\n
project_root <- find_project_root()\n
setwd(project_root)\n
cat("Working directory:", getwd(), "\\n")\n

In [ ]:
scripts <- c("ddop_data.R", "ddop_utility.R", "cv.R", "fwfs.R", "bwfs.R", "genetic.R", "ddop_plot.R")\n
missing_scripts <- scripts[!file.exists(scripts)]\n
if (length(missing_scripts) > 0) stop("Script mancanti: ", paste(missing_scripts, collapse = ", "))\n
invisible(lapply(scripts, source))\n

In [ ]:
normalize_target <- function(x) {\n
  chr <- tolower(trimws(as.character(x)))\n
\n
  if (all(chr %in% c("no", "yes", NA_character_))) {\n
    return(factor(chr, levels = c("no", "yes")))\n
  }\n
\n
  if (all(chr %in% c("0", "1", NA_character_))) {\n
    mapped <- ifelse(chr == "1", "yes", ifelse(chr == "0", "no", NA_character_))\n
    return(factor(mapped, levels = c("no", "yes")))\n
  }\n
\n
  levels_found <- sort(unique(na.omit(chr)))\n
  if (length(levels_found) == 2) {\n
    mapped <- ifelse(chr == levels_found[1], "no", ifelse(chr == levels_found[2], "yes", NA_character_))\n
    warning("`euro_d` rimappata automaticamente a no/yes da: ", paste(levels_found, collapse = ", "))\n
    return(factor(mapped, levels = c("no", "yes")))\n
  }\n
\n
  stop("`euro_d` non in formato binario. Valori trovati: ", paste(head(levels_found, 10), collapse = ", "))\n
}\n
\n
dataset_path <- file.path("data", "dataset.sav")\n
stopifnot(file.exists(dataset_path))\n
dataset <- haven::read_sav(dataset_path)\n
\n
if (!"euro_d" %in% names(dataset)) stop("La variabile target `euro_d` non e presente nel dataset.")\n
dataset$euro_d <- normalize_target(dataset$euro_d)\n
\n
cat("Rows:", nrow(dataset), " Cols:", ncol(dataset), "\\n")\n
dataset %>% count(euro_d) %>% mutate(perc = round(n / sum(n) * 100, 2))\n

In [ ]:
na_profile <- var_na_perc(dataset)\n
na_profile\n
\n
top_missing <- sort(colMeans(is.na(dataset)) * 100, decreasing = TRUE)\n
head(top_missing, 20)\n
\n
domain_mapping <- get_domain_map(dataset)\n
var_types <- get_var_type(dataset, domain_mapping)\n
\n
tibble::tibble(\n
  gruppo = c("continue", "ord", "not_ord", "euro"),\n
  n = c(\n
    length(var_types$var_continue),\n
    length(var_types$var_ord),\n
    length(var_types$var_not_ord),\n
    length(var_types$var_euro)\n
  )\n
)\n

In [ ]:
# CV rapida: piu leggera della pipeline completa\n
results_cv_quick <- cv(dataset, model_types = c("glm"), seed = 123, k_folds = 5)\n
\n
cv_metrics <- dplyr::bind_rows(results_cv_quick$results)\n
cv_summary <- cv_metrics %>%\n
  group_by(Model, Indicator) %>%\n
  summarise(\n
    Mean = mean(Value, na.rm = TRUE),\n
    SD = sd(Value, na.rm = TRUE),\n
    Min = min(Value, na.rm = TRUE),\n
    Max = max(Value, na.rm = TRUE),\n
    .groups = "drop"\n
  ) %>%\n
  arrange(Model, Indicator)\n
\n
cv_summary\n

In [ ]:
best_thresholds <- results_cv_quick$thresholds_results %>%\n
  group_by(Model) %>%\n
  slice_max(order_by = Accuracy, n = 1, with_ties = FALSE) %>%\n
  ungroup() %>%\n
  select(Model, Threshold, Accuracy, Sensitivity, Specificity, PPV, NPV, Population_below_threshold)\n
\n
best_thresholds\n
\n
ggplot(results_cv_quick$thresholds_results, aes(x = Threshold, y = Accuracy, color = Model)) +\n
  geom_line(linewidth = 1) +\n
  geom_point(size = 1.8) +\n
  labs(title = "Accuracy vs Threshold", x = "Threshold (%)", y = "Accuracy") +\n
  theme_minimal(base_size = 12)\n

In [ ]:
# Esecuzione estesa (runtime lungo):\n
# results_cv <- cv(dataset)\n
# results_cv_under <- double_cv(dataset, balance = "under")\n
# results_cv_over <- double_cv(dataset, balance = "over")\n
# results_fwfs <- random_forward_selection(dataset)\n
# results_bwfs <- random_backward_selection(dataset)\n
# results_ga <- genetic_feature_selection(dataset)\n